# transformer-inference-lab — MQA training run (n_kv_head=1)

Trains the MQA checkpoint (`n_kv_head=1`) — the last of the three variants, for the memory–latency–quality comparison against MHA (n_kv_head=8) and GQA (n_kv_head=2). Run as **Save & Run All (Commit)** — should be the fastest of the three, ~2.3-2.7h on a T4.

**IMPORTANT — this notebook trains MQA, config `configs/mqa.yaml`, n_kv_head MUST be 1.** A previous run got mislabeled (file named mqa-training but content was unchanged from the GQA notebook, so it silently trained GQA again). Cell 6 below asserts n_kv_head==1 before training starts — if it fails, STOP and check which config actually got cloned.

**Before running:** attach `transformer-inference-lab-data` (sidebar → Add Input). Confirm `configs/mqa.yaml` has `max_iters: 5000` on GitHub.

## 1. GPU check + clone repo

In [ ]:
!nvidia-smi
!git clone https://github.com/modestesavadogo/transformer-inference-lab.git
%cd /kaggle/working/transformer-inference-lab
!git log --oneline -8

## 2. Install dependencies

In [ ]:
!pip install -r requirements.txt --quiet

## 3. Link the dataset
Requires `transformer-inference-lab-data` attached via the notebook's Input sidebar.

In [ ]:
!mkdir -p data
!ln -s /kaggle/input/datasets/modestesavadogomaths/transformer-inference-lab-data/train.bin data/train.bin
!ln -s /kaggle/input/datasets/modestesavadogomaths/transformer-inference-lab-data/val.bin data/val.bin
!ls -la data/
!test -f data/train.bin && echo "train.bin target exists" || echo "train.bin target MISSING"
!test -f data/val.bin && echo "val.bin target exists" || echo "val.bin target MISSING"

## 4. Verify data before committing to a multi-hour run

In [ ]:
import numpy as np

train_data = np.memmap('data/train.bin', dtype=np.uint16, mode='r')
val_data = np.memmap('data/val.bin', dtype=np.uint16, mode='r')
print(f"train: {len(train_data):,} tokens")
print(f"val:   {len(val_data):,} tokens")

assert len(train_data) > 40_000_000, "train.bin looks too small — check dataset link"
assert len(val_data) > 10_000, "val.bin looks too small — check dataset link"
print("data check passed")

## 5. Confirm config — MUST show n_kv_head: 1
This is the exact check that would have caught the previous mislabeling. Do not proceed past the assertion failing.

In [ ]:
!cat configs/mqa.yaml

In [ ]:
import yaml

with open("configs/mqa.yaml") as f:
    cfg = yaml.safe_load(f)

n_kv_head = cfg["model"]["n_kv_head"]
print(f"n_kv_head = {n_kv_head}")
assert n_kv_head == 1, (
    f"WRONG CONFIG: expected n_kv_head=1 for MQA, got {n_kv_head}. "
    "This is the exact bug that mislabeled the last run as GQA. STOP."
)
print("confirmed: this is genuinely the MQA config")

## 6. Run tests — cheap insurance before a multi-hour job

In [ ]:
!python -m pytest tests/ -q

## 7. Train MQA
Runs to completion in the background under Save & Run All, even if the tab is closed.

In [ ]:
!python train.py --config configs/mqa.yaml --device cuda --eval-interval 250 --log-interval 50

## 8. Confirm the checkpoint is valid AND is genuinely MQA
Double-checks n_kv_head in the saved checkpoint's config, not just the yaml file — catches the case where the yaml was right but train.py somehow loaded a stale config.

In [ ]:
import torch

ckpt = torch.load("results/checkpoints/mqa.pt", map_location="cuda")
print("iter:", ckpt["iter"])
print("val_loss:", ckpt.get("val_loss"))
print("config:", ckpt["config"])
print("num tensors in state dict:", len(ckpt["model_state_dict"]))

assert ckpt.get("val_loss") is not None, "val_loss missing — train.py fix may not have been pulled"
assert ckpt["config"]["n_kv_head"] == 1, (
    f"CHECKPOINT IS NOT MQA: n_kv_head={ckpt['config']['n_kv_head']} in saved checkpoint. "
    "Do not upload this as mqa.pt."
)
print("confirmed: checkpoint is genuinely MQA (n_kv_head=1)")

## 9. Upload checkpoint as a Kaggle Dataset
Same pattern as MHA and GQA — do this before the session recycles.

In [ ]:
!mkdir -p checkpoint_upload
!cp results/checkpoints/mqa.pt checkpoint_upload/

In [ ]:
import json

metadata = {
    "title": "transformer-inference-lab-checkpoints",
    "id": "modestesavadogomaths/transformer-inference-lab-checkpoints",
    "licenses": [{"name": "CC0-1.0"}]
}
with open("checkpoint_upload/dataset-metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
print(open("checkpoint_upload/dataset-metadata.json").read())

In [ ]:
# dataset already exists (mha.pt, gqa.pt already in it) — use 'version' not 'create'
!kaggle datasets version -p checkpoint_upload/ -m "add mqa.pt checkpoint" --dir-mode zip

## 10. Summary — all three checkpoints now exist
Print a final comparison line for easy BUILDLOG copy-paste.

In [ ]:
print("MQA training complete.")
print(f"  iter:     {ckpt['iter']}")
print(f"  val_loss: {ckpt['val_loss']:.4f}")
print(f"  n_kv_head: {ckpt['config']['n_kv_head']}")
print()
print("Compare against:")
print("  MHA val_loss: 5.6314")
print("  GQA val_loss: 5.7637")